# How I Use Bifrost MCP Gateway and Code Mode to Give Claude Code Safer MCP Access with Lower Token Cost

Companion notebook for [the complete To Data & Beyond tutorial](https://todatabeyond.com/blog/how-i-use-bifrost-mcp-gateway-and-code-mode-to-give-claude-code-safer-mcp-access-with-lower-token-cost). View the [maintained notebook on GitHub](https://github.com/To-Data-Beyond/Generative-AI-Techanical-Tutorials/blob/main/Bifrost_MCP_Gateway_and_Code_Mode.ipynb).


## Before you begin

This companion is intentionally documentation-first. Run the commands in a local terminal where Bifrost, Docker, and Claude Code are installed. Review every path and permission before execution.

This notebook intentionally contains no saved execution outputs or credentials.


MCP becomes useful very quickly once you connect Claude Code to real tools.

At first, that feels like a clear win. Claude Code can do more than generate text. It can reach external systems, call tools, and work against a richer environment. The problem is that once the number of MCP tools starts growing, two practical issues show up fast: governance and cost.

The governance problem is straightforward. In a real setup, not every agent session should be allowed to access every tool. Once you connect multiple MCP servers, you need a way to control which tools are exposed, how access is scoped, and what gets logged.

Bifrost’s MCP Gateway is designed around that layer. The docs and the engineering blog both position it as a governed /mcp endpoint in front of your MCP tool ecosystem, with controls like virtual keys, tool filtering, and auditability.

The second problem is less obvious at first, but it matters a lot once you scale the tool count. Classic MCP setups can become expensive because tool definitions get pushed into the model context repeatedly. Bifrost Code Mode changes that execution pattern by letting the model discover tools through lightweight interfaces and execute short tool-orchestration code instead of carrying the full tool surface directly in the prompt context every time. The post reports large token reductions in its benchmark as tool count increases, including reductions above 90% in later rounds.

That is what made this setup interesting to me.

I was not only looking for a way to connect Claude Code to MCP tools. I was looking for a workflow that felt closer to production use: one where tool access can be scoped, MCP calls can be governed through a gateway, and token overhead does not keep growing with every additional tool.

In this post, I’ll walk through that setup step by step using Bifrost MCP Gateway and Code Mode with Claude Code. I’ll show the actual configuration flow, what each setting does, and what the output looks like after each step so you can reproduce it on your side.

## Table of Contents

---

## 1. Add Your First MCP Client in Bifrost

The first practical step is to register an MCP server inside Bifrost. This is where Bifrost begins to serve as the control layer in front of your MCP tools. In the dashboard, you can add an MCP client by giving it a name, choosing a connection type (such as HTTP, SSE, or STDIO), and entering the endpoint or command. For HTTP and SSE servers, you can also attach any required headers directly in the UI, such as API keys, auth tokens, or custom metadata. Once the client is saved, Bifrost connects to it, discovers its available tools, and shows it in the MCP client list with a live health indicator.

You can start by opening the MCP section in the Bifrost dashboard. To get to Bifrost dashboard, you will need first to run this command in your terminal:


**Command or configuration**

```bash
npx -y @maximhq/bifrost
```


Then you go to your browser and open:


**Command or configuration**

```bash
http://localhost:8080
```


Once you open the dashboard you should go to the MCP gateway option and it should look similar to the figure below.

![Bifrost dashboard with the MCP Gateway catalog selected](https://todatabeyond.com/articles/bifrost-mcp-gateway/dashboard-mcp-section.png)

*The MCP section is where Bifrost manages upstream MCP servers before exposing them through the gateway*

You can now click Add MCP Client. From there, you can fill in the basic connection details for the MCP server you want Bifrost to manage. If the upstream server is already exposed over HTTP, I would use the server URL. If it runs through SSE, I would use the SSE endpoint. If it is a local MCP server that should be launched directly by Bifrost, I would choose STDIO and provide the startup command. Bifrost supports all three patterns, so the right choice depends on how your MCP server is deployed.

For my first MCP client, I started with a simple local STDIO setup using the filesystem server. In the Bifrost MCP Gateway, I created a new server named filesystem, selected STDIO as the connection type, set the command to npx, and passed the arguments -y, @modelcontextprotocol/server-filesystem followed by an allowed local directory path. I also exposed the HOME and PATH environment variables so the launched process could resolve the local environment correctly.

Before saving that configuration in Bifrost, I verified the same command directly in the terminal to make sure the MCP server could actually start successfully. That gave me a reliable baseline before moving on to Code Mode and tool-governance settings.

![Bifrost form for adding a filesystem MCP client over STDIO](https://todatabeyond.com/articles/bifrost-mcp-gateway/add-mcp-client.png)

*When adding a new MCP client, Bifrost lets you choose the connection type and configure the upstream server details directly in the UI.*

What matters at this stage is not Claude Code yet. The goal here is to get the MCP server registered inside Bifrost first, because everything that comes later—Code Mode, auto-execute rules, virtual-key scoping, and the final Claude Code connection—depends on Bifrost already knowing about this MCP client and its tools.

After saving the client, I would verify three things before moving on:

- The MCP client appears in the list,
- The health indicator shows that the connection is live
- The tools from the upstream server have been discovered correctly.

![Bifrost MCP server catalog showing a connected filesystem client and discovered tools](https://todatabeyond.com/articles/bifrost-mcp-gateway/mcp-client-health.png)

*Once the client is saved, Bifrost connects to the server, discovers its tools, and tracks its health from the MCP client list*

If any of those fail, the first things I would check are the endpoint or command, whether the server is actually running, and whether any required headers were entered correctly. Bifrost also refreshes connected clients periodically, which helps it pick up new tools from upstream MCP servers over time.

This step is simple, but it is foundational. Once the MCP client is connected here, Bifrost has a real tool surface to govern, expose, and optimize in the next steps.

---

## 2. Enable Code Mode for the MCP Client

Once the MCP client is connected successfully, the next step is to enable Code Mode for that client.

This setting changes how the client’s tools are exposed to the model. With Code Mode enabled, the tools from that MCP client are no longer injected directly into the default tool list.

Instead, they become accessible through four generic meta-tools: listToolFiles, readToolFile, getToolDocs, and executeToolCode. Bifrost describes this as a way to keep the tool surface much smaller in context while still letting the model discover the tools it needs and execute short orchestration code inside a constrained sandbox.

In the dashboard, this step is simple. I open the MCP Gateway page, click the connected client row to open its configuration sheet, then toggle Code Mode Client to enabled and save the change. Bifrost documents Code Mode as a per-client setting, which means I can decide client by client which MCP servers should use this execution pattern and which ones should remain in the classic mode. That is useful because not every MCP server needs Code Mode from the start.

![Bifrost filesystem MCP client configuration with Code Mode enabled](https://todatabeyond.com/articles/bifrost-mcp-gateway/enable-code-mode.png)

*Code Mode is enabled per MCP client, which makes it possible to mix classic MCP and Code Mode across different servers*

What I like about doing this after the initial connection is that it keeps the setup easier to debug. First, I confirm that the MCP server can connect normally. Then, once that baseline is working, I enable Code Mode and verify that the client is now being handled through the Code Mode path instead of the regular direct-tool path. That sequence made the setup much clearer in practice.

Code Mode becomes especially useful once the number of MCP tools starts growing. Bifrost’s documentation recommends it for cases such as having 3+ MCP servers, complex multi-step workflows, or concern about token cost and latency. The same page also notes that small setups with only one or two lightweight servers may still be fine with classic MCP, and that both modes can be mixed depending on the server.

So in my setup, this step is where the workflow starts shifting from “Bifrost as a connection layer” to “Bifrost as an execution and optimization layer.” The client is already connected at this point. Enabling Code Mode changes how Claude Code will interact with that client later through the gateway.

---

## 3. Understand What Code Mode Actually Changes

After enabling Code Mode, the MCP client still stays connected in Bifrost, but the way its tools are exposed changes completely.

In a classic MCP setup, the model sees the tool definitions directly and can call those tools one by one as part of the normal tool-calling loop. With Code Mode, Bifrost stops exposing that client’s tools in the default tool list. Instead, it gives the model access to four generic meta-tools: listToolFiles, readToolFile, getToolDocs, and executeToolCode. Through those four tools, the model can first discover what is available, then read compact tool signatures and documentation on demand, and finally submit a short piece of code that Bifrost executes in a constrained Starlark sandbox.

That difference matters because it changes both the size of the tool surface and the execution flow. Bifrost’s documentation explains that, rather than sending large catalogs of tool definitions on every turn, Code Mode keeps only the four meta-tools in context and moves the actual orchestration into the sandbox. The intermediate steps happen there, and the model receives a much smaller final result instead of carrying every tool definition and every intermediate output through repeated turns.

The best way to think about it is this: classic MCP is mostly tool calling directly from the model, while Code Mode is closer to the model writing a short orchestration script for the tools it needs. Bifrost documents this explicitly as “AI writes Python to orchestrate tools,” with the execution happening in a sandboxed Starlark environment.

This is also why Code Mode tends to become more useful as the number of MCP servers and tools increases. Bifrost’s docs give a concrete comparison: with around five MCP servers and about one hundred tools, classic MCP keeps all tool definitions in context across multiple turns, while Code Mode reduces that down to the four generic tools plus only the tool information that is read on demand. The docs summarize the result as about 50% lower cost and 3–4x fewer LLM round trips in that example flow. The page also recommends Code Mode when you have 3+ MCP servers, complex multi-step workflows, or concern about cost and latency.

In practice, this section is where the Bifrost setup starts to feel more opinionated. I am no longer just routing Claude Code to a collection of MCP tools. I am also choosing an execution pattern that is designed to scale better as the tool surface grows. That is the main reason I wanted to get Code Mode working before connecting Claude Code to the gateway itself.

---

## 4. Configure Which Tools Can Auto-Execute

Once Code Mode was enabled for the filesystem client, the next step was deciding which tools should be allowed to run automatically.

Bifrost exposed the full tool list from the filesystem MCP server, including both read-oriented tools and write-capable ones. I kept all tools enabled, but I did not allow all of them to auto-execute.

![Bifrost filesystem MCP client showing fourteen available tools before auto-execution is configured](https://todatabeyond.com/articles/bifrost-mcp-gateway/filesystem-tool-list.png)

*After connecting the filesystem MCP client, Bifrost exposes the full tool list so auto-execution can be configured tool by tool.*

Instead, I started with a safer default by auto-allowing only the tools that are mainly used for inspection and retrieval: list_allowed_directories, list_directory, list_directory_with_sizes, directory_tree, get_file_info, search_files, read_text_file, and read_multiple_files.

![Bifrost filesystem tools with eight read-oriented operations enabled for automatic execution](https://todatabeyond.com/articles/bifrost-mcp-gateway/auto-execute-tools.png)

*I started with a safer default by auto-allowing read and inspection tools while keeping file-changing operations behind approval.*

For tools that can modify the local filesystem, I stayed more conservative. I left operations like create_directory, edit_file, and move_file outside the auto-execute set, which means they remain available but still require approval before they run. For an initial setup, that felt like the right balance. Claude Code could explore and understand the filesystem efficiently, but anything that could change files or directories stayed gated behind a confirmation step.

This was one of the parts I liked most in the Bifrost MCP Gateway workflow. I was not only connecting Claude Code to MCP tools. I was also deciding which parts of that tool surface were safe enough to automate and which ones should remain controlled. That made the setup feel much closer to a real governed workflow than simply exposing every tool and letting the agent run freely.

---

## 5. Scope Access with Virtual Keys

Once the MCP client was connected and its auto-execute policy was in place, the next step was to scope access with a virtual key. This is the point where Bifrost starts acting less like a simple MCP connector and more like a governed gateway.

In Bifrost, virtual keys can carry their own MCP tool-access configuration, and the filtering docs are explicit about the default behavior: if a virtual key has no MCP configuration, then no MCP tools are available through that key. In other words, MCP access through virtual keys is deny by default until you explicitly allow specific clients and tools.

![Bifrost Virtual Keys page for creating scoped gateway credentials](https://todatabeyond.com/articles/bifrost-mcp-gateway/virtual-keys.png)

*Virtual keys are where Bifrost scopes access, budgets, and governance for each consumer of the gateway.*

In practice, that means I do not have to expose the full filesystem MCP client to every consumer of the gateway. I can create a dedicated virtual key for Claude Code, then attach only the MCP client and tools I actually want that key to access.

Bifrost supports this directly from the dashboard by letting you create or edit a virtual key and then add MCP Client Configurations for the allowed clients and tools. The same filtering layer can be applied at the tool level, not only at the server level, which is exactly what makes the workflow useful for real agent setups.

For my setup, this is where I would create a dedicated virtual key for the Claude Code integration and attach only the filesystem client with the read-oriented tools I had already decided were safe enough to expose. That keeps the access scope aligned with the earlier auto-execution step. Claude Code still gets the filesystem capabilities I want it to have, but not the entire MCP surface by default.

![Bifrost virtual key configuration for selecting an MCP client and allowed tools](https://todatabeyond.com/articles/bifrost-mcp-gateway/virtual-key-mcp-access.png)

*MCP access is configured directly on the virtual key by selecting which MCP clients and tools that key is allowed to use*

Bifrost also supports MCP Tool Groups, which are named collections of tools that can be attached to virtual keys, teams, customers, or providers. That becomes more useful once the setup grows beyond a single MCP client, because it lets you define a reusable tool policy once and apply it in more than one place. For this first practical setup, though, a single virtual key with explicit MCP client and tool configuration is the clearest way to see how the scoping works.

![Bifrost virtual key list showing the active Claude Code MCP credential](https://todatabeyond.com/articles/bifrost-mcp-gateway/filtered-tools.png)

*With virtual-key filtering enabled, only the explicitly allowed MCP clients and tools are exposed through that key*

This step is one of the main reasons I wanted to go through the gateway instead of connecting Claude Code directly to MCP servers. The question is no longer just “can Claude Code reach the tool?” It becomes “which key is being used, and exactly which tools is that key allowed to call?” That is a much better model for controlled agent access.

---

## 6. Connect Claude Code to the Bifrost MCP Gateway

Once the MCP client was connected, Code Mode was enabled, and access was scoped through a virtual key, the final step was to connect Claude Code to the Bifrost MCP Gateway itself.

Bifrost exposes the gateway through a single MCP endpoint:


**Command or configuration**

```bash
http://localhost:8080/mcp
```


That is the endpoint Claude Code connects to. When a virtual key is used, it is passed in the request headers, and Bifrost uses that key to decide which MCP tools should be visible to Claude Code. In other words, Claude Code does not connect directly to the upstream filesystem server. It connects to Bifrost, and Bifrost becomes the governed layer in front of the MCP tool surface.

In practice, this step is very simple. From the terminal, I add the Bifrost MCP Gateway to Claude Code and include the virtual key in the headers:


**Command or configuration**

```bash
claude mcp add-json bifrost '{
  "type": "http",
  "url": "http://localhost:8080/mcp",
  "headers": {
    "Authorization": "Bearer YOUR_VIRTUAL_KEY"
  }
}'
```


Bifrost’s Claude Code and MCP docs show this exact pattern: Claude Code points to the /mcp endpoint, and the virtual key can be passed through the Authorization header to scope access.

If I do not need key-based scoping, Claude Code can also be connected without the header:


**Command or configuration**

```bash
claude mcp add-json bifrost '{
  "type": "http",
  "url": "http://localhost:8080/mcp"
}'
```


But for this tutorial, I prefer using the virtual key because it keeps the setup aligned with the governed-access workflow from the earlier steps. That way, the tools Claude Code sees are not simply “whatever is connected to Bifrost.” They are only the tools allowed by the key I created for this integration.

After running the command, the easiest way to validate the setup is to open Claude Code and check whether the MCP tools appear as expected. At this stage, Claude Code should see the Bifrost-exposed tool surface rather than the raw upstream MCP client directly. If the virtual key was scoped correctly, only the allowed tools should be visible. If the gateway was connected without a key, then visibility depends on the broader Bifrost configuration.

This is the point where the full workflow comes together. The filesystem server is still the actual upstream MCP server. But Claude Code is no longer talking to it directly. Claude Code is talking to Bifrost, and Bifrost is now handling connection, filtering, Code Mode behavior, and access control in one place. That is what makes the setup much more usable as the tool surface grows.

---

## 7. Test the Workflow End to End

Once Claude Code is connected to the Bifrost MCP Gateway, the last step is to test the workflow from end to end.

At this point, the main thing I want to verify is not only that Claude Code can see MCP tools. I want to confirm the whole chain is working the way I configured it: Claude Code talks to the Bifrost /mcp endpoint, Bifrost applies the virtual-key scope, the filesystem client stays behind the gateway, and Code Mode handles the interaction through its meta-tool flow instead of exposing the full raw tool surface directly.

Bifrost’s MCP docs describe that gateway pattern explicitly, and the Code Mode docs explain that Code Mode clients are surfaced through the generic discovery and execution flow rather than the classic direct-tool path.

For this first test, I would start with a simple read-oriented prompt inside Claude Code. Since the virtual key and auto-execute policy were scoped around safe filesystem inspection, I would avoid starting with anything that modifies files. A good first prompt would be something like:


**Prompt**

```prompt
Show me the files available in the allowed directory and summarize the folder structure.
```


That kind of request is useful because it exercises the exact behavior I set up earlier. Claude Code should be able to inspect the allowed directory through the gateway, use the filesystem MCP client indirectly, and return a result without needing any write-capable tool. If everything is configured correctly, the request should stay within the safe read-oriented path I allowed in the earlier steps. The filesystem server package itself is built around scoped filesystem access to allowed paths, which is why testing directory inspection first is a good sanity check.

After that, I would run one or two slightly more specific prompts, for example:


**Prompt**

```prompt
Read the text files in the allowed folder and tell me what each one contain
```


or


**Prompt**

```prompt
Search for files related to "report" in the allowed directory and summarize what you find
```


These kinds of prompts help verify that the retrieval-oriented tools are actually reachable through Claude Code, while still staying inside the safer tool subset. If those prompts work cleanly, that is a good sign that the gateway connection, the virtual-key scoping, and the auto-execute settings are aligned.

The next thing I would check is what Claude Code cannot do. This is just as important as confirming what it can do. For example, if I deliberately ask Claude Code to rename a file or edit a file, I should not expect that to run automatically if those write-capable tools were left outside the auto-execute set or excluded from the virtual-key scope.

That negative test is useful because it confirms the gateway is actually enforcing the boundaries I configured rather than simply exposing the whole MCP client without restriction. Bifrost’s filtering and governance docs make clear that visibility and execution are shaped by the configured MCP access policy and tool-level controls.

If I want to validate the governance side more explicitly, I would also open the Bifrost dashboard after a few Claude Code requests and check the logs or MCP views to confirm that the calls are flowing through the gateway rather than bypassing it. Bifrost positions the gateway as the governance and observability layer in front of MCP clients, so this is a useful final confirmation step in the tutorial.

This end-to-end test is where the whole setup becomes tangible. Claude Code still feels like Claude Code from the user side, but behind the scenes the interaction is now governed by Bifrost: the MCP server stays behind one gateway, the accessible tool surface is scoped by the virtual key, and Code Mode keeps the execution pattern more controlled as the tool surface grows. That is the point of the whole workflow.

---

## 8. What This Setup Gives Me in Practice

After putting Claude Code behind the Bifrost MCP Gateway, the main difference for me is not just that MCP tools are available. It is that the workflow becomes much easier to control.

Instead of connecting Claude Code directly to an MCP server and exposing the whole tool surface at once, I now have a gateway layer in the middle. That changes the setup in a few useful ways. First, access becomes easier to scope. The tools Claude Code can see are no longer just a function of which MCP server exists. They depend on the virtual key and the filtering rules attached to it. That makes the tool surface much more intentional.

Second, the setup becomes easier to reason about from a safety perspective. In my case, I started with a filesystem MCP client, but I did not want Claude Code to automatically run every filesystem operation. With Bifrost in the middle, I could keep the fast path for read-oriented tools while leaving write-capable operations behind approval. That is a much better default than treating all tools as equally safe.

Third, Code Mode gives this setup a better execution pattern as the tool surface grows. Instead of carrying the full raw tool list directly in the normal MCP flow, Code Mode reduces that down to a smaller generic interface for discovery and execution. That matters more as more MCP servers and more tools get added over time. Bifrost’s Code Mode docs explicitly position it as the better fit for larger MCP setups, especially when cost and latency start becoming real concerns.

What I like most is that none of this changes the user-facing experience too much. Claude Code still feels like Claude Code. The difference is in the layer underneath: the MCP server is now behind one gateway, access is filtered through a virtual key, and the execution path is more governed than a direct one-off MCP connection. That makes the setup feel much closer to something I would actually want to use in a larger or more serious environment.
